In [ ]:
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
# %load_ext viztracer
# from viztracer import VizTracer
%load_ext autoreload
%autoreload 3
import sys
from pathlib import Path

# required to enable non-blocking interaction:
%gui qt5

from copy import deepcopy
# from numba import jit
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
# pd.options.mode.dtype_backend = 'pyarrow' # use new pyarrow backend instead of numpy
from attrs import define, field, fields, Factory
import tables as tb
from datetime import datetime, timedelta

# Pho's Formatting Preferences
import builtins

import IPython
from IPython.core.formatters import PlainTextFormatter
from IPython import get_ipython

from pyphocorehelpers.preferences_helpers import set_pho_preferences, set_pho_preferences_concise, set_pho_preferences_verbose
set_pho_preferences_concise()
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# BEGIN PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #


## IPython pprint
from pyphocorehelpers.pprint import wide_pprint, wide_pprint_ipython, wide_pprint_jupyter, MAX_LINE_LENGTH

# Override default pprint
builtins.pprint = wide_pprint

text_formatter: PlainTextFormatter = IPython.get_ipython().display_formatter.formatters['text/plain']
text_formatter.max_width = MAX_LINE_LENGTH
text_formatter.for_type(object, wide_pprint_jupyter)

import IPython
from IPython.core.formatters import PlainTextFormatter
from IPython import get_ipython
from IPython.display import display
# from pho_jupyter_preview_widget.display_helpers import array_repr_with_graphical_preview
from pho_jupyter_preview_widget.ipython_helpers import PreviewWidgetMagics

ip = get_ipython()

# Register the magic
ip.register_magics(PreviewWidgetMagics)

# # from pyphocorehelpers.ipython_helpers import MyMagics
# # %config_ndarray_preview width=500

# # Register the custom display function for NumPy arrays
# ip = array_repr_with_graphical_preview(ip=ip)



# END PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

from pyphocorehelpers.print_helpers import get_now_time_str, get_now_day_str

## Pho's Custom Libraries:
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager

# NeuroPy (Diba Lab Python Repo) Loading
# from neuropy import core
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from typing_extensions import TypeAlias
from nptyping import NDArray
import neuropy.utils.type_aliases as types

from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.core.epoch import NamedTimerange, Epoch
from neuropy.core.ratemap import Ratemap
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder
from neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat import KDibaOldDataSessionFormatRegisteredClass
from neuropy.utils.matplotlib_helpers import matplotlib_file_only, matplotlib_configuration, matplotlib_configuration_update
from neuropy.core.neuron_identities import NeuronIdentityTable, neuronTypesList, neuronTypesEnum
from neuropy.utils.mixins.AttrsClassHelpers import AttrsBasedClassHelperMixin, serialized_field, serialized_attribute_field, non_serialized_field, custom_define
from neuropy.utils.mixins.HDF5_representable import HDF_DeserializationMixin, post_deserialize, HDF_SerializationMixin, HDFMixin, HDF_Converter

## For computation parameters:
from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.utils.dynamic_container import DynamicContainer
from neuropy.utils.result_context import IdentifyingContext
from neuropy.core.session.Formats.BaseDataSessionFormats import find_local_session_paths
from neuropy.core.neurons import NeuronType
from neuropy.core.user_annotations import UserAnnotationsManager
from neuropy.core.position import Position
from neuropy.core.session.dataSession import DataSession
from neuropy.analyses.time_dependent_placefields import PfND_TimeDependent, PlacefieldSnapshot
from neuropy.utils.debug_helpers import debug_print_placefield, debug_print_subsession_neuron_differences, debug_print_ratemap, debug_print_spike_counts, debug_plot_2d_binning, print_aligned_columns
from neuropy.utils.debug_helpers import parameter_sweeps, _plot_parameter_sweep, compare_placefields_info
from neuropy.utils.indexing_helpers import NumpyHelpers, union_of_arrays, intersection_of_arrays, find_desired_sort_indicies, paired_incremental_sorting
from pyphocorehelpers.print_helpers import print_object_memory_usage, print_dataframe_memory_usage, print_value_overview_only, DocumentationFilePrinter, print_keys_if_possible, generate_html_string, document_active_variables

## Pho Programming Helpers:
import inspect
from pyphocorehelpers.print_helpers import DocumentationFilePrinter, TypePrintMode, print_keys_if_possible, debug_dump_object_member_shapes, print_value_overview_only, document_active_variables
from pyphocorehelpers.programming_helpers import IPythonHelpers, PythonDictionaryDefinitionFormat, MemoryManagement, inspect_callable_arguments, get_arguments_as_optional_dict, GeneratedClassDefinitionType, CodeConversion
from pyphocorehelpers.gui.Qt.TopLevelWindowHelper import TopLevelWindowHelper, print_widget_hierarchy
from pyphocorehelpers.indexing_helpers import reorder_columns, reorder_columns_relative, dict_to_full_array
from pyphocorehelpers.programming_helpers import CodeConversion, SourceCodeParsing, GeneratedClassDefinitionType

# doc_output_parent_folder: Path = Path('EXTERNAL/DEVELOPER_NOTES/DataStructureDocumentation').resolve() # ../.
# print(f"doc_output_parent_folder: {doc_output_parent_folder}")
# assert doc_output_parent_folder.exists()

# pyPhoPlaceCellAnalysis:
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline # get_neuron_identities
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import export_pyqtgraph_plot
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_load_session, batch_extended_computations, batch_extended_programmatic_figures
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import PipelineSavingScheme

import pyphoplacecellanalysis.External.pyqtgraph as pg

from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_perform_all_plots
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import JonathanFiringRateAnalysisResult
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import _find_any_context_neurons
from pyphoplacecellanalysis.General.Batch.runBatch import BatchSessionCompletionHandler # for `post_compute_validate(...)`
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import AcrossSessionsResults
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends # for `_perform_long_short_instantaneous_spike_rate_groups_analysis`
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import SingleBarResult, InstantaneousSpikeRateGroupsComputation, TruncationCheckingResults # for `BatchSessionCompletionHandler`, `AcrossSessionsAggregator`
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import SplitPartitionMembership
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalPlacefieldGlobalComputationFunctions, DirectionalLapsResult, TrackTemplates
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.RankOrderComputations import RankOrderGlobalComputationFunctions
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import TrackTemplates
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.RankOrderComputations import RankOrderComputationsContainer, RankOrderResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.RankOrderComputations import RankOrderAnalyses


# Plotting
# import pylustrator # customization of figures
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
_bak_rcParams = mpl.rcParams.copy()

matplotlib.use('Qt5Agg')
# %matplotlib inline
# %matplotlib auto

# _restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')
_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

from silx.gui import qt
from silx.gui.plot import Plot2D, Plot1D
from silx.gui.colors import Colormap
from silx.gui.plot.items import ImageBase

# import pylustrator # call `pylustrator.start()` before creating your first figure in code.
from pyphoplacecellanalysis.Pho2D.matplotlib.visualize_heatmap import visualize_heatmap
from pyphoplacecellanalysis.Pho2D.matplotlib.visualize_heatmap import visualize_heatmap_pyqtgraph # used in `plot_kourosh_activity_style_figure`
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import plot_multiple_raster_plot, plot_raster_plot
from pyphoplacecellanalysis.General.Mixins.DataSeriesColorHelpers import UnitColoringMode, DataSeriesColorHelpers
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import _build_default_tick, build_scatter_plot_kwargs
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.Render2DScrollWindowPlot import Render2DScrollWindowPlotMixin, ScatterItemData
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_extended_programmatic_figures, batch_programmatic_figures
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends
from pyphoplacecellanalysis.General.Mixins.SpikesRenderingBaseMixin import SpikeEmphasisState

from pyphoplacecellanalysis.SpecificResults.PhoDiba2023Paper import PAPER_FIGURE_figure_1_add_replay_epoch_rasters, PAPER_FIGURE_figure_1_full, PAPER_FIGURE_figure_3, main_complete_figure_generations
from pyphoplacecellanalysis.SpecificResults.fourthYearPresentation import *
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import SerializationHelper_AllCustomDecodingResults, SerializationHelper_CustomDecodingResults
    
# Jupyter Widget Interactive
import ipywidgets as widgets
from IPython.display import display, HTML
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import interactive_pipeline_widget, interactive_pipeline_files
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget, render_colors

from datetime import datetime, date, timedelta
from pyphocorehelpers.print_helpers import get_now_day_str, get_now_rounded_time_str

DAY_DATE_STR: str = date.today().strftime("%Y-%m-%d")
DAY_DATE_TO_USE = f'{DAY_DATE_STR}' # used for filenames throught the notebook
print(f'DAY_DATE_STR: {DAY_DATE_STR}, DAY_DATE_TO_USE: {DAY_DATE_TO_USE}')

NOW_DATETIME: str = get_now_rounded_time_str()
NOW_DATETIME_TO_USE = f'{NOW_DATETIME}' # used for filenames throught the notebook
print(f'NOW_DATETIME: {NOW_DATETIME}, NOW_DATETIME_TO_USE: {NOW_DATETIME_TO_USE}')


from pyphocorehelpers.gui.Jupyter.simple_widgets import build_global_data_root_parent_path_selection_widget
all_paths = [Path('/Volumes/SwapSSD/Data'), Path('/Users/pho/data'), Path(r'/media/MAX/Data'), Path(r'/media/halechr/MAX/Data'), Path(r'/home/halechr/FastData'), Path(r'W:\Data'), Path(r'/home/halechr/cloud/turbo/Data'), Path(r'/Volumes/MoverNew/data'), Path(r'/home/halechr/turbo/Data')]
global_data_root_parent_path = None
def on_user_update_path_selection(new_path: Path):
    global global_data_root_parent_path
    new_global_data_root_parent_path = new_path.resolve()
    global_data_root_parent_path = new_global_data_root_parent_path
    print(f'global_data_root_parent_path changed to {global_data_root_parent_path}')
    assert global_data_root_parent_path.exists(), f"global_data_root_parent_path: {global_data_root_parent_path} does not exist! Is the right computer's config commented out above?"
            
global_data_root_parent_path_widget = build_global_data_root_parent_path_selection_widget(all_paths, on_user_update_path_selection)
global_data_root_parent_path_widget

In [ ]:
import pyphoplacecellanalysis.External.pyqtgraph as pg
from pyphoplacecellanalysis.External.pyqtgraph.Qt import QtGui, QtCore, QtWidgets
# from pyphoplacecellanalysis.External.pyqtgraph.parametertree.parameterTypes.file import popupFilePicker
from pyphoplacecellanalysis.External.pyqtgraph.widgets.FileDialog import FileDialog

from silx.gui import qt
from silx.gui.dialog.ImageFileDialog import ImageFileDialog
from silx.gui.dialog.DataFileDialog import DataFileDialog
import silx.io

from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import saveFile

from pyphoplacecellanalysis.General.Pipeline.Stages.Loading import loadData

from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DecoderDecodedEpochsResult
from pyphoplacecellanalysis.GUI.Silx.EpochHeuristicPosteriorDebuggerWidget import EpochHeuristicDebugger

# load_path = Path('W:/Data/KDIBA/gor01/one/2006-6-09_1-22-43/output/2024-11-25_CustomDecodingResults.pkl').resolve()
# load_path = Path(r"W:\Data\KDIBA\gor01\one\2006-6-08_14-26-15\output\2024-05-06_CustomDecodingResults.pkl").resolve() # new relative format

# load_path = Path('/media/halechr/MAX/Data/KDIBA/gor01/one/2006-6-08_14-26-15/output/2024-08-01_CustomDecodingResults.pkl').resolve() # LAB

# load_path = Path('W:/Data/KDIBA/gor01/one/2006-6-09_1-22-43/output/2024-11-25_CustomDecodingResults.pkl').resolve()
# directional_decoders_epochs_decode_result, xbin, xbin_centers =  SerializationHelper_CustomDecodingResults.load(load_path=load_path)


# load_path = Path("W:/Data/KDIBA/gor01/one/2006-6-09_1-22-43/output/2024-11-25_AllCustomDecodingResults.pkl")
# load_path = Path("W:/Data/KDIBA/gor01/one/2006-6-09_1-22-43/output/2025-06-16_AllCustomDecodingResults.pkl")

load_path = Path("W:/Data/KDIBA/gor01/one/2006-6-09_1-22-43/output/2025-06-16_AllCustomDecodingResults.pkl")


track_templates, directional_decoders_epochs_decode_result, xbin, xbin_centers, extras =  SerializationHelper_AllCustomDecodingResults.load(load_path=load_path)
pos_bin_size = directional_decoders_epochs_decode_result.pos_bin_size
long_pf2D = extras.get('long_pf2D')
print(f'extras: {list(extras.keys())}')
print(f'extras: {list(extras.keys())}')
a_decoded_filter_epochs_decoder_result_dict = extras.pop('a_decoded_filter_epochs_decoder_result_dict')
heuristic_new_scores = extras.pop('_out_new_scores')
high_heuristic_only_filtered_decoder_filter_epochs_decoder_result_dict = extras.pop('high_heuristic_only_filtered_decoder_filter_epochs_decoder_result_dict')
# heuristic_new_scores

# 2026-10-01 - Standalone Cursor-produced example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def wcorr(arr: np.ndarray) -> float:
    """Same estimator as neuropy.analyses.decoders.wcorr. arr is (n_positions, n_time_bins)."""
    nx, ny = arr.shape[1], arr.shape[0]
    y_mat: np.ndarray = np.tile(np.arange(ny)[:, np.newaxis], (1, nx))
    x_mat: np.ndarray = np.tile(np.arange(nx), (ny, 1))
    arr_sum: float = np.nansum(arr)
    ey: float = np.nansum(arr * y_mat) / arr_sum
    ex: float = np.nansum(arr * x_mat) / arr_sum
    cov_xy: float = np.nansum(arr * (y_mat - ey) * (x_mat - ex)) / arr_sum
    cov_yy: float = np.nansum(arr * (y_mat - ey) ** 2) / arr_sum
    cov_xx: float = np.nansum(arr * (x_mat - ex) ** 2) / arr_sum
    return cov_xy / np.sqrt(cov_xx * cov_yy)


def gaussian_columns(n_pos: int, mu: np.ndarray, sigma: float) -> np.ndarray:
    """Column-normalized Gaussian posterior. mu shape (n_time,)."""
    y = np.arange(n_pos)[:, np.newaxis]
    mu_row = np.asarray(mu, dtype=float)[np.newaxis, :]
    log_mass = -0.5 * ((y - mu_row) / sigma) ** 2
    log_mass = log_mass - np.max(log_mass, axis=0, keepdims=True)
    posterior = np.exp(log_mass)
    posterior /= np.sum(posterior, axis=0, keepdims=True)
    return posterior


# Synthetic decoded posterior. Columns sum to 1, including the flat bins outside the sweep.
N_POS = 60
N_TIME = 100
TRUE_START = 30
TRUE_END = 62  # exclusive
THRESHOLD = 0.7  # min_wcorr_threshold in PhoDiba2023Paper
MIN_BINS = 6
CROP = 8
PAD = 4
BIN_S = 0.025  # illustrative decoding bin; 4 bins = 100 ms

posterior = np.full((N_POS, N_TIME), 1.0 / N_POS)
posterior[:, TRUE_START:TRUE_END] = gaussian_columns(N_POS, np.linspace(4.0, 55.0, TRUE_END - TRUE_START), sigma=1.25)


def score_window(start: int, end: int) -> float:
    if end - start < 2 or start < 0 or end > N_TIME:
        return np.nan
    with np.errstate(invalid='ignore', divide='ignore'):
        value = float(wcorr(posterior[:, start:end]))
    if not np.isfinite(value):
        return np.nan
    return value


score_grid = np.full((N_TIME, N_TIME + 1), np.nan)
for start in range(0, N_TIME - MIN_BINS):
    for end in range(start + MIN_BINS, N_TIME + 1):
        score_grid[start, end] = score_window(start, end)
    ## END for end in range(start + MIN_BINS, N_TIME + 1)...

## END for start in range(0, N_TIME - MIN_BINS)...

extra_bins = np.arange(-12, 13)
score_if_start_moves = np.array([score_window(TRUE_START - int(k), TRUE_END) for k in extra_bins])
score_if_end_moves = np.array([score_window(TRUE_START, TRUE_END + int(k)) for k in extra_bins])

windows = {
    'true': (TRUE_START, TRUE_END),
    'shorter': (TRUE_START + CROP, TRUE_END - CROP),
    'wider': (TRUE_START - PAD, TRUE_END + PAD),
}
window_scores = {name: score_window(start, end) for name, (start, end) in windows.items()}

both_edges_fail_at = next(k for k in range(0, 16) if score_window(TRUE_START - k, TRUE_END + k) < THRESHOLD)
print(f"true    [{TRUE_START}, {TRUE_END}):  wcorr = {window_scores['true']:.3f}")
print(f"shorter [{TRUE_START + CROP}, {TRUE_END - CROP}):  wcorr = {window_scores['shorter']:.3f}   (still above {THRESHOLD})")
print(f"wider   [{TRUE_START - PAD}, {TRUE_END + PAD}):  wcorr = {window_scores['wider']:.3f}   (below {THRESHOLD})")
print(f"padding both edges by {both_edges_fail_at} bins drops wcorr below {THRESHOLD}  ({both_edges_fail_at * BIN_S * 1e3:.0f} ms at {BIN_S * 1e3:.0f} ms/bin)")

vmax = float(np.max(posterior))
cmap = plt.cm.magma.copy()
cmap.set_bad('#e6e6e6')

fig = plt.figure(figsize=(12.8, 9.4), layout='constrained')
grid = fig.add_gridspec(3, 6, height_ratios=[1.25, 1.05, 1.35])
ax_post = fig.add_subplot(grid[0, :])
ax_short = fig.add_subplot(grid[1, 0:2], sharey=ax_post)
ax_true = fig.add_subplot(grid[1, 2:4], sharey=ax_post)
ax_wide = fig.add_subplot(grid[1, 4:6], sharey=ax_post)
ax_map = fig.add_subplot(grid[2, 0:3])
ax_curves = fig.add_subplot(grid[2, 3:6])

ax_post.imshow(posterior, origin='lower', aspect='auto', interpolation='nearest', vmin=0.0, vmax=vmax, cmap='magma', extent=(-0.5, N_TIME - 0.5, -0.5, N_POS - 0.5))
map_pos = np.argmax(posterior, axis=0).astype(float)
map_pos[:TRUE_START] = np.nan
map_pos[TRUE_END:] = np.nan
ax_post.plot(np.arange(N_TIME), map_pos, color='white', lw=1.1)
for edge in (TRUE_START, TRUE_END - 1):
    ax_post.axvline(edge, color='white', lw=1.2)
## END for edge in (TRUE_START, TRUE_END - 1)...
ax_post.axvline(TRUE_START - PAD, color='tab:orange', ls='--', lw=1.1)
ax_post.axvline(TRUE_END - 1 + PAD, color='tab:orange', ls='--', lw=1.1)
ax_post.set_xlim(-0.5, N_TIME - 0.5)
ax_post.set_ylabel('position bin')
ax_post.set_xlabel('time bin')
ax_post.set_title('Decoded posterior. White lines: true sweep. Orange dashed: 4 flat bins added on each side. Flat columns still sum to 1.')


def show_slice(ax, start: int, end: int, title: str) -> None:
    ax.imshow(posterior[:, start:end], origin='lower', aspect='auto', interpolation='nearest', vmin=0.0, vmax=vmax, cmap='magma', extent=(start - 0.5, end - 0.5, -0.5, N_POS - 0.5))
    value = score_window(start, end)
    relation = 'above' if value >= THRESHOLD else 'below'
    ax.set_title(f"{title}\n[{start}, {end})   wcorr = {value:.2f}  ({relation} {THRESHOLD})", fontsize=10)
    ax.set_xlabel('time bin')


show_slice(ax_short, *windows['shorter'], f"{CROP} bins trimmed from each side")
show_slice(ax_true, *windows['true'], "true epoch")
show_slice(ax_wide, *windows['wider'], f"{PAD} flat bins added on each side")
ax_short.set_ylabel('position bin')

masked = np.ma.masked_invalid(score_grid)
image = ax_map.imshow(masked, origin='lower', aspect='auto', interpolation='nearest', vmin=0.0, vmax=1.0, cmap=cmap, extent=(-0.5, N_TIME + 0.5, -0.5, N_TIME - 0.5))
ax_map.contour(np.arange(N_TIME + 1), np.arange(N_TIME), masked, levels=[THRESHOLD], colors='cyan', linewidths=1.3)
ax_map.plot(TRUE_END, TRUE_START, marker='*', ms=16, color='white', markeredgecolor='black', linestyle='none', label='true (start, end)', zorder=3)
ax_map.plot(TRUE_END + PAD, TRUE_START - PAD, marker='o', ms=7, color='tab:orange', markeredgecolor='black', linestyle='none', label=f'{PAD} extra bins each side', zorder=3)
ax_map.set_xlim(TRUE_START - 22, TRUE_END + 22)
ax_map.set_ylim(TRUE_START - 22, TRUE_END - MIN_BINS)
ax_map.set_xlabel('epoch end bin (exclusive)')
ax_map.set_ylabel('epoch start bin')
ax_map.set_title(f"wcorr for every (start, end). Cyan contour is {THRESHOLD}.")
ax_map.legend(loc='upper right', frameon=True, fontsize=8)
fig.colorbar(image, ax=ax_map, fraction=0.046, pad=0.04, label='wcorr')

ax_curves.axvspan(0, extra_bins[-1], color='tab:orange', alpha=0.12, label='window longer than the sweep')
ax_curves.axvspan(extra_bins[0], 0, color='0.5', alpha=0.10, label='window shorter than the sweep')
ax_curves.axhline(THRESHOLD, color='0.15', ls='--', lw=1.0, label=f'threshold {THRESHOLD}')
ax_curves.axvline(0.0, color='0.25', lw=0.8)
ax_curves.plot(extra_bins, score_if_start_moves, marker='o', ms=3.5, label='shift the start, end fixed')
ax_curves.plot(extra_bins, score_if_end_moves, marker='s', ms=3.5, label='shift the end, start fixed')
ax_curves.set_xlim(extra_bins[0], extra_bins[-1])
ax_curves.set_ylim(-0.05, 1.05)
ax_curves.set_xlabel('extra bins beyond the true bound   (negative = truncated)')
ax_curves.set_ylabel('wcorr')
ax_curves.set_title('Moving either bound')
ax_curves.legend(loc='lower right', fontsize=8, frameon=False)

fig.suptitle('Weighted correlation stays high on every piece of the sweep and drops when the epoch includes flat bins', fontsize=12)
plt.show()

# Resume pre 2026-10-01